# Section 6: Post-Metadata Dataset Audit and Visual Inspection
Audits true image validity, geometries, basic quality, and defines candidates for review.
Does not alter the operational manifest. Does not modify raw images.

In [1]:
import pandas as pd
import numpy as np
import os
from PIL import Image
import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUTPUT_ROOT   = r"C:\SKIN CANCER v2\pipe output"
manifest_path = os.path.join(OUTPUT_ROOT, "manifests", "training_eligible_manifest_post_metadata.csv")

d_audit = os.path.join(OUTPUT_ROOT, "audit_reports", "image_quality")
os.makedirs(d_audit, exist_ok=True)

## Load Post-Metadata Manifest

In [2]:
if not os.path.exists(manifest_path):
    raise FileNotFoundError(f"Manifest not found: {manifest_path}")

df = pd.read_csv(manifest_path)
total_rows   = len(df)
class_counts = df["final_authoritative_label"].value_counts()

print(f"Loaded post-metadata manifest: {total_rows:,} rows")
print("Class distribution:")
for cls in ["NV", "MEL", "BCC"]:
    print(f"  {cls}: {int(class_counts.get(cls, 0)):,}")

Loaded post-metadata manifest: 20,513 rows
Class distribution:
  NV: 12,736
  MEL: 4,468
  BCC: 3,309


## True Image Decode Validation & Basic Quality Extraction

In [3]:
def extract_image_specs(row):
    path  = row["full_path"]
    specs = {
        "image_decode_success":    False,
        "decode_error_message":    "None",
        "decoded_width":           0,
        "decoded_height":          0,
        "decoded_channels":        0,
        "brightness_mean":         np.nan,
        "dark_pixel_fraction":     np.nan,
        "contrast_std":            np.nan,
        "sharpness_laplacian_var": np.nan
    }
    try:
        img = cv2.imread(path)
        if img is None:
            specs["decode_error_message"] = "cv2.imread returned None"
            return specs
        if len(img.shape) == 2:
            h, w, c = img.shape[0], img.shape[1], 1
            gray = img
        else:
            h, w, c = img.shape
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        specs.update({
            "image_decode_success":    True,
            "decoded_width":           w,
            "decoded_height":          h,
            "decoded_channels":        c,
            "brightness_mean":         float(gray.mean()),
            "dark_pixel_fraction":     float((gray < 30).mean()),
            "contrast_std":            float(gray.std()),
            "sharpness_laplacian_var": float(cv2.Laplacian(gray, cv2.CV_64F).var())
        })
    except Exception as e:
        specs["decode_error_message"] = str(e)
    return specs

print(f"Decoding {total_rows:,} images...")
decoded_features = df.apply(extract_image_specs, axis=1).tolist()
df_features = pd.DataFrame(decoded_features)
df_full     = pd.concat([df.reset_index(drop=True), df_features.reset_index(drop=True)], axis=1)
df_full["aspect_ratio"] = df_full["decoded_width"] / df_full["decoded_height"].replace(0, 1)

n_ok   = int((df_full["image_decode_success"] == True).sum())
n_fail = int((df_full["image_decode_success"] == False).sum())
print(f"Decode successes: {n_ok:,}")
print(f"Decode failures : {n_fail:,}")

Decoding 20,513 images...
Decode successes: 20,513
Decode failures : 0


## Decode & Geometry Summary

In [4]:
failed_decode = df_full[df_full["image_decode_success"] == False]
df_valid      = df_full[df_full["image_decode_success"] == True].copy()
df_valid["aspect_ratio"] = df_valid["decoded_width"] / df_valid["decoded_height"]

geo_summary = df_valid[["decoded_width","decoded_height","aspect_ratio"]].describe()
print("=== OVERALL GEOMETRY SUMMARY ===")
display(geo_summary)

class_geo_summary = (
    df_valid.groupby("final_authoritative_label")[["decoded_width","decoded_height","aspect_ratio"]]
    .agg(["mean","median","min","max"])
)
display(class_geo_summary)
class_geo_summary.to_csv(os.path.join(d_audit, "image_dimension_summary_by_class.csv"))

=== OVERALL GEOMETRY SUMMARY ===


,decoded_width,decoded_height,aspect_ratio
count,20513.000000,20513.000000,20513.000000
mean,850.254765,754.274606,1.185597
std,207.797729,270.164838,0.183621
min,576.000000,450.000000,0.750000
25%,600.000000,450.000000,1.000000
50%,1024.000000,768.000000,1.333333
75%,1024.000000,1024.000000,1.333333
max,1024.000000,1024.000000,1.546474


decoded_width                    decoded_height  \
                                   mean  median  min   max           mean   
final_authoritative_label                                                   
BCC                          958.138410  1024.0  600  1024     934.838320   
MEL                          916.578782  1024.0  600  1024     844.288272   
NV                           798.957443   600.0  576  1024     675.783213   

                                             aspect_ratio                      \
                           median  min   max         mean    median       min   
final_authoritative_label                                                       
BCC                        1024.0  450  1024     1.051778  1.000000  1.000000   
MEL                        1024.0  450  1024     1.133113  1.000000  0.836914   
NV                          450.0  450  1024     1.238778  1.333333  0.750000   

                                     
                                max  
final_authoritative_label            
BCC                        1.333333  
MEL                        1.546474  
NV                         1.530643

## Quality Metrics Summary & Candidate Generation

In [5]:
q_summary = df_valid[
    ["brightness_mean","dark_pixel_fraction","contrast_std","sharpness_laplacian_var"]
].describe()
print("=== RAW QUALITY METRICS SUMMARY ===")
display(q_summary)

# Data-driven thresholds
width_low_pct     = float(df_valid["decoded_width"].quantile(0.01))
height_low_pct    = float(df_valid["decoded_height"].quantile(0.01))
ar_low_pct        = float(df_valid["aspect_ratio"].quantile(0.005))
ar_high_pct       = float(df_valid["aspect_ratio"].quantile(0.995))
bright_low_pct    = float(df_valid["brightness_mean"].quantile(0.01))
bright_high_pct   = float(df_valid["brightness_mean"].quantile(0.99))
dark_high_pct     = float(df_valid["dark_pixel_fraction"].quantile(0.99))
contrast_low_pct  = float(df_valid["contrast_std"].quantile(0.02))
contrast_vlow_pct = float(df_valid["contrast_std"].quantile(0.005))
sharp_low_pct     = float(df_valid["sharpness_laplacian_var"].quantile(0.02))
sharp_vlow_pct    = float(df_valid["sharpness_laplacian_var"].quantile(0.005))

threshold_table = pd.DataFrame([
    {"metric": "decoded_width_low_1pct",         "threshold": width_low_pct},
    {"metric": "decoded_height_low_1pct",        "threshold": height_low_pct},
    {"metric": "aspect_ratio_low_0.5pct",        "threshold": ar_low_pct},
    {"metric": "aspect_ratio_high_99.5pct",      "threshold": ar_high_pct},
    {"metric": "brightness_low_1pct",            "threshold": bright_low_pct},
    {"metric": "brightness_high_99pct",          "threshold": bright_high_pct},
    {"metric": "dark_pixel_fraction_high_99pct", "threshold": dark_high_pct},
    {"metric": "contrast_std_low_2pct",          "threshold": contrast_low_pct},
    {"metric": "contrast_std_low_0.5pct",        "threshold": contrast_vlow_pct},
    {"metric": "sharpness_low_2pct",             "threshold": sharp_low_pct},
    {"metric": "sharpness_low_0.5pct",           "threshold": sharp_vlow_pct},
])
threshold_table.to_csv(os.path.join(d_audit, "quality_flag_thresholds.csv"), index=False)
print("=== DATA-DRIVEN QUALITY THRESHOLDS ===")
display(threshold_table)

=== RAW QUALITY METRICS SUMMARY ===


,brightness_mean,dark_pixel_fraction,contrast_std,sharpness_laplacian_var
count,20513.000000,20513.000000,20513.000000,20513.000000
mean,146.420805,0.059941,36.783200,65.265387
std,29.798365,0.149823,20.288529,124.356156
min,32.365700,0.000000,5.403519,1.973822
25%,132.664121,0.000000,21.479815,21.811211
50%,150.198045,0.000085,31.401118,36.078198
75%,166.434644,0.015407,46.699533,65.416462
max,239.581470,0.708912,105.918514,3825.024580


=== DATA-DRIVEN QUALITY THRESHOLDS ===


,metric,threshold
0,decoded_width_low_1pct,600.000000
1,decoded_height_low_1pct,450.000000
2,aspect_ratio_low_0.5pct,1.000000
3,aspect_ratio_high_99.5pct,1.508100
4,brightness_low_1pct,55.297780
5,brightness_high_99pct,203.455897
6,dark_pixel_fraction_high_99pct,0.667284
7,contrast_std_low_2pct,10.784860
8,contrast_std_low_0.5pct,8.605974
9,sharpness_low_2pct,9.231696


In [6]:
# Flag conditions
cond_decode        = df_full["image_decode_success"] == False
cond_small_dim     = (df_full["decoded_width"] < width_low_pct) | (df_full["decoded_height"] < height_low_pct)
cond_extreme_ar    = (df_full["image_decode_success"] == True) & (
    (df_full["aspect_ratio"] < ar_low_pct) | (df_full["aspect_ratio"] > ar_high_pct))
cond_extreme_bri   = (df_full["brightness_mean"] < bright_low_pct) | (df_full["brightness_mean"] > bright_high_pct)
cond_too_dark      = df_full["dark_pixel_fraction"] > dark_high_pct
cond_low_contrast  = df_full["contrast_std"] < contrast_low_pct
cond_vlow_contrast = df_full["contrast_std"] < contrast_vlow_pct
cond_low_sharp     = df_full["sharpness_laplacian_var"] < sharp_low_pct
cond_vlow_sharp    = df_full["sharpness_laplacian_var"] < sharp_vlow_pct

soft_flag_count = (
    cond_extreme_bri.astype(int) + cond_too_dark.astype(int) +
    cond_low_contrast.astype(int) + cond_low_sharp.astype(int)
)
candidate_mask = (
    cond_decode | cond_small_dim | cond_extreme_ar |
    (soft_flag_count >= 2) | cond_vlow_contrast | cond_vlow_sharp
)
review_candidates = df_full[candidate_mask].copy()

def build_flag_reasons(row):
    reasons = []
    if not row["image_decode_success"]:            reasons.append("decode_failure")
    if row["decoded_width"] < width_low_pct or row["decoded_height"] < height_low_pct:
        reasons.append("small_dimensions")
    if row["image_decode_success"] and (
        row["aspect_ratio"] < ar_low_pct or row["aspect_ratio"] > ar_high_pct):
        reasons.append("extreme_aspect_ratio")
    if pd.notna(row["brightness_mean"]) and (
        row["brightness_mean"] < bright_low_pct or row["brightness_mean"] > bright_high_pct):
        reasons.append("extreme_brightness")
    if pd.notna(row["dark_pixel_fraction"]) and row["dark_pixel_fraction"] > dark_high_pct:
        reasons.append("too_dark")
    if pd.notna(row["contrast_std"]):
        if row["contrast_std"] < contrast_vlow_pct:   reasons.append("very_low_contrast")
        elif row["contrast_std"] < contrast_low_pct:  reasons.append("low_contrast")
    if pd.notna(row["sharpness_laplacian_var"]):
        if row["sharpness_laplacian_var"] < sharp_vlow_pct:  reasons.append("very_low_sharpness")
        elif row["sharpness_laplacian_var"] < sharp_low_pct: reasons.append("low_sharpness")
    return "|".join(reasons) if reasons else "none"

review_candidates["flag_reasons"]    = review_candidates.apply(build_flag_reasons, axis=1)
review_candidates["soft_flag_count"] = soft_flag_count.loc[review_candidates.index]

flag_reason_counts = (
    review_candidates["flag_reasons"].str.split("|", regex=False).explode()
    .value_counts().rename_axis("flag_reason").reset_index(name="count")
)
candidate_counts_by_class = (
    review_candidates["final_authoritative_label"].value_counts()
    .rename_axis("class_label").reset_index(name="candidate_count")
)

print(f"\nTotal quality review candidates: {len(review_candidates):,}")
print(f"Candidate percentage: {round(len(review_candidates)/len(df_full)*100,2)}%")
print("=== BY CLASS ===")
display(candidate_counts_by_class)
print("=== BY FLAG REASON ===")
display(flag_reason_counts)

review_candidates.to_csv(os.path.join(d_audit, "quality_review_candidates.csv"), index=False)
candidate_counts_by_class.to_csv(os.path.join(d_audit, "quality_candidate_counts_by_class.csv"), index=False)
flag_reason_counts.to_csv(os.path.join(d_audit, "quality_candidate_counts_by_reason.csv"), index=False)


Total quality review candidates: 458
Candidate percentage: 2.23%
=== BY CLASS ===


,class_label,candidate_count
0,NV,220
1,BCC,144
2,MEL,94


=== BY FLAG REASON ===


,flag_reason,count
0,extreme_brightness,147
1,too_dark,132
2,very_low_contrast,103
3,very_low_sharpness,103
4,extreme_aspect_ratio,91
5,low_sharpness,62
6,low_contrast,15
7,small_dimensions,5


## Generate Class Sample Grids

In [7]:
def generate_grid(sample_df, title, save_path, labels=True):
    if len(sample_df) == 0:
        return
    n          = min(9, len(sample_df))
    sample_df  = sample_df.sample(n, random_state=42)
    fig, axes  = plt.subplots(3, 3, figsize=(10, 10))
    fig.suptitle(title, fontsize=16)
    for ax, (_, row) in zip(axes.flatten(), sample_df.iterrows()):
        path = row["full_path"]
        if os.path.exists(path):
            img = cv2.imread(path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            ax.imshow(img)
        ax.axis("off")
        if labels:
            ax.set_title(row["final_authoritative_label"])
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

print("Building sample grids...")
for cls in ["NV", "MEL", "BCC"]:
    generate_grid(
        df_valid[df_valid["final_authoritative_label"] == cls],
        f"Sample Grid: {cls}",
        os.path.join(d_audit, f"class_sample_grid_{cls}.png")
    )
generate_grid(df_valid, "Mixed Unlabeled Audit Grid",
              os.path.join(d_audit, "mixed_unlabeled_sample_grid.png"), labels=False)
print("Grids saved.")

Building sample grids...
Grids saved.


## Export Files

In [8]:
df_full.to_csv(os.path.join(d_audit, "image_decode_validation_report.csv"), index=False)
geo_summary.rename_axis("statistic").to_csv(os.path.join(d_audit, "image_dimension_summary.csv"))
q_summary.rename_axis("statistic").to_csv(os.path.join(d_audit, "image_quality_summary.csv"))

pd.DataFrame([{
    "total_rows":        total_rows,
    "decode_failures":   n_fail,
    "valid_decoded_rows": len(df_valid),
    "review_candidates": len(review_candidates),
    "NV_count":          int(class_counts.get("NV", 0)),
    "MEL_count":         int(class_counts.get("MEL", 0)),
    "BCC_count":         int(class_counts.get("BCC", 0))
}]).to_csv(os.path.join(d_audit, "post_metadata_dataset_summary.csv"), index=False)

# NOTE: quality_review_candidates.csv requires a manual_quality_decision column before
# running the quality exclusion step below. Populate: keep | exclude | uncertain
# then run the cell that follows.
print("Export complete. Review quality_review_candidates.csv and populate manual_quality_decision.")

Export complete. Review quality_review_candidates.csv and populate manual_quality_decision.


## Apply Manual Quality Exclusion (run after filling manual_quality_decision)

In [11]:
manifest_dir       = os.path.join(OUTPUT_ROOT, "manifests")
image_quality_dir  = d_audit
source_manifest_path = os.path.join(manifest_dir, "training_eligible_manifest_post_metadata.csv")
quality_review_path  = os.path.join(image_quality_dir, "quality_review_candidates.csv")
output_manifest_path = os.path.join(manifest_dir, "training_eligible_manifest_post_quality.csv")
dropped_rows_path    = os.path.join(OUTPUT_ROOT, "audit_reports", "quality_rows_dropped.csv")
quality_summary_path = os.path.join(OUTPUT_ROOT, "audit_reports", "quality_exclusion_summary.csv")

df_source = pd.read_csv(source_manifest_path)
df_review = pd.read_csv(quality_review_path)

if "manual_quality_decision" not in df_review.columns:
    print("manual_quality_decision column not yet present in quality_review_candidates.csv.")
    print("Populate that column (keep | exclude | uncertain) then rerun this cell.")
else:
    valid_decisions = {"keep", "exclude", "uncertain"}
    bad_decisions   = set(df_review["manual_quality_decision"].dropna().unique()) - valid_decisions
    if bad_decisions:
        raise ValueError(f"Unexpected manual_quality_decision values: {bad_decisions}")

    df_exclude    = df_review[df_review["manual_quality_decision"] == "exclude"].copy()
    exclude_paths = set(df_exclude["full_path"].tolist())
    missing_from_source = df_exclude[~df_exclude["full_path"].isin(df_source["full_path"])]
    if len(missing_from_source) > 0:
        raise ValueError(f"{len(missing_from_source)} excluded rows not found in source manifest.")

    df_post_quality = df_source[~df_source["full_path"].isin(exclude_paths)].copy()
    df_dropped      = df_source[df_source["full_path"].isin(exclude_paths)].copy()
    review_cols = [c for c in ["full_path","manual_quality_decision","manual_quality_reason",
                                "flag_reasons","review_pass","reviewer_notes"] if c in df_review.columns]
    df_dropped  = df_dropped.merge(
        df_review[review_cols].drop_duplicates(subset=["full_path"]),
        on="full_path", how="left"
    )

    df_post_quality.to_csv(output_manifest_path, index=False)
    df_dropped.to_csv(dropped_rows_path, index=False)
    post_class_counts = df_post_quality["final_authoritative_label"].value_counts()
    pd.DataFrame([{
        "source_manifest":    "training_eligible_manifest_post_metadata.csv",
        "output_manifest":    "training_eligible_manifest_post_quality.csv",
        "source_row_count":   len(df_source),
        "dropped_row_count":  len(df_dropped),
        "output_row_count":   len(df_post_quality),
        "NV_count":           int(post_class_counts.get("NV", 0)),
        "MEL_count":          int(post_class_counts.get("MEL", 0)),
        "BCC_count":          int(post_class_counts.get("BCC", 0))
    }]).to_csv(quality_summary_path, index=False)

    print(f"Source rows  : {len(df_source):,}")
    print(f"Dropped rows : {len(df_dropped):,}")
    print(f"Remaining    : {len(df_post_quality):,}")
    print("Post-quality class counts:")
    for cls in ["NV","MEL","BCC"]:
        print(f"  {cls}: {int(post_class_counts.get(cls, 0)):,}")

Source rows  : 20,513
Dropped rows : 124
Remaining    : 20,389
Post-quality class counts:
  NV: 12,708
  MEL: 4,430
  BCC: 3,251


In [12]:
# Final summary
_dec_fail   = int(n_fail)
_cands      = len(review_candidates)
_has_pq     = os.path.exists(os.path.join(OUTPUT_ROOT, "manifests", "training_eligible_manifest_post_quality.csv"))

print("=" * 60)
print("  06_post_metadata_dataset_audit -- FINAL SUMMARY")
print("=" * 60)
print(f"\nTotal rows checked           : {total_rows:,}")
print(f"Decode failures              : {_dec_fail:,}")
print(f"Quality review candidates    : {_cands:,} ({round(_cands/total_rows*100,2)}%)")
print(f"\nQuality flag counts by reason:")
for _, row in flag_reason_counts.iterrows():
    print(f"  {row['flag_reason']:<28}: {row['count']:,}")
print(f"\nClass counts (post-metadata manifest):")
for cls in ["NV","MEL","BCC"]:
    print(f"  {cls}: {int(class_counts.get(cls, 0)):,}")
print(f"\nPost-quality manifest created: {_has_pq}")
print(f"Raw images modified          : False (read-only access)")
print(f"\nOutput file verification:")
_check_files = [
    os.path.join(d_audit, "quality_review_candidates.csv"),
    os.path.join(d_audit, "quality_candidate_counts_by_class.csv"),
    os.path.join(d_audit, "quality_candidate_counts_by_reason.csv"),
    os.path.join(d_audit, "image_decode_validation_report.csv"),
    os.path.join(d_audit, "quality_flag_thresholds.csv"),
]
for p in _check_files:
    exists = os.path.exists(p)
    size   = os.path.getsize(p) if exists else 0
    status = "OK" if exists else "MISSING"
    print(f"  [{status}] {os.path.basename(p):<45} {size:>10,} bytes")
print("=" * 60)

  06_post_metadata_dataset_audit -- FINAL SUMMARY

Total rows checked           : 20,513
Decode failures              : 0
Quality review candidates    : 458 (2.23%)

Quality flag counts by reason:
  extreme_brightness          : 147
  too_dark                    : 132
  very_low_contrast           : 103
  very_low_sharpness          : 103
  extreme_aspect_ratio        : 91
  low_sharpness               : 62
  low_contrast                : 15
  small_dimensions            : 5

Class counts (post-metadata manifest):
  NV: 12,736
  MEL: 4,468
  BCC: 3,309

Post-quality manifest created: True
Raw images modified          : False (read-only access)

Output file verification:
  [OK] quality_review_candidates.csv                    204,784 bytes
  [OK] quality_candidate_counts_by_class.csv                 54 bytes
  [OK] quality_candidate_counts_by_reason.csv               184 bytes
  [OK] image_decode_validation_report.csv             8,939,140 bytes
  [OK] quality_flag_thresholds.csv       